# Training the setting classifier

Fine-tunes [`lkonle/fiction-gbert-large`](https://huggingface.co/lkonle/fiction-gbert-large) on `data/space_annotations.csv` (70/30 split, learning rate 1e-5, 5 epochs, best epoch kept).
Needs a GPU (the original model was trained on Google Colab).

In [ ]:
# !pip install transformers datasets accelerate scikit-learn

In [ ]:
from pathlib import Path

import numpy as np
from datasets import ClassLabel, load_dataset
from sklearn.metrics import classification_report, f1_score
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          Trainer, TrainingArguments)

ANNOTATIONS = "../data/space_annotations.csv"
BASE_MODEL = "lkonle/fiction-gbert-large"
OUTPUT_DIR = "setting-classifier"

LABELS = ["perceived_space", "action_space", "visual_space", "descriptive_space", "no_space"]
label2id = {label: i for i, label in enumerate(LABELS)}
id2label = dict(enumerate(LABELS))

In [ ]:
ds = load_dataset("csv", data_files=ANNOTATIONS)["train"]
ds = ds.filter(lambda x: x["space_type"] in LABELS)
ds = ds.cast_column("space_type", ClassLabel(names=LABELS))
ds = ds.remove_columns(["filename", "sentence_id"]).rename_column("space_type", "label")

# the split for the published model was not seeded, so numbers will differ slightly
ds = ds.train_test_split(test_size=0.3, seed=42)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
ds = ds.map(lambda batch: tokenizer(batch["text"], truncation=True), batched=True)
ds

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    print(classification_report(labels, preds, target_names=LABELS, digits=2, zero_division=0))
    per_class = f1_score(labels, preds, average=None, labels=range(len(LABELS)), zero_division=0)
    return {"f1_micro": f1_score(labels, preds, average="micro"),
            **{f"f1_{label}": f for label, f in zip(LABELS, per_class)}}

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=len(LABELS), label2id=label2id, id2label=id2label)

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=1e-5,
    num_train_epochs=5,
    per_device_train_batch_size=14,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["test"],
                  data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)
trainer.train()

## Evaluation of the best checkpoint (Table 2)

In [ ]:
trainer.evaluate()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)